In [1]:
#------------------------------------------------ Begin_Librairie ----------------------------------------
import datetime
import os
import time
import re
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from time import sleep
from urllib.parse import urljoin
from docx import Document
import win32com.client as win32
import warnings
warnings.filterwarnings('ignore', message='Boolean Series key will be reindexed')

In [2]:
# install on the control website library code.
# Optional: pip install openpyxl python-docx pywin32
# (.doc SLA lists need Microsoft Word + pywin32; .docx SLA lists need python-docx)
# import subprocess, sys
# subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'openpyxl', 'python-docx', 'pywin32'])


In [3]:
#------------------------------------------------ Begin_ fileName ----------------------------------------
regulatorName = 'MD NBMO'

print(f"Running {regulatorName} Web Scraping Tool v.1.1")
now = datetime.datetime.now()
filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(':', '.')[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"
os.chdir(scriptfolder)
tempfolder = os.path.join(scriptfolder, 'tempfolder')
if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)


Running MD NBMO Web Scraping Tool v.1.1


In [4]:
#------------------------------------------------ Begin_chromedriver ----------------------------------------
chromeOptions = webdriver.ChromeOptions()
prefs = {
    'plugins.always_open_pdf_externally': True,
    'download.prompt_for_download': False,
    'download.default_directory': tempfolder,
    'profile.default_content_setting_values.automatic_downloads': 1,
}
chromeOptions.add_experimental_option('prefs', prefs)


In [5]:
#------------------------------------------------ Begin_Variable ----------------------------------------

processdate = now.strftime('%Y-%m-%d')
regdict={
	'MD NBMO 1': 'https://www.bnm.md/en/content/authorized-banks-republic-moldova',
	'MD NBMO 2': 'https://www.bnm.md/en/content/supervised-entities-insurance-and-non-bank-lending#art1',
	'MD NBMO 3': 'https://www.bnm.md/en/content/supervised-entities-insurance-and-non-bank-lending#art1',
	'MD NBMO 4': 'https://www.bnm.md/en/content/supervised-entities-insurance-and-non-bank-lending#art1',
	'MD NBMO 5': 'https://www.bnm.md/en/content/supervised-entities-insurance-and-non-bank-lending#art1',
		 }

xlsx_xpath = {
	'2': [
		"//a[contains(@href, 'RPPPA_Asiguratori') and contains(@href, '.xlsx')]",
		"//a[contains(@href, 'brokerilor') and contains(@href, '.xlsx')]",
		"//a[contains(@href, 'bancassurance') and contains(@href, '.xlsx')]",
	],
	'3': ["//a[contains(@href, 'ROCNA_BNM_web') and contains(@href, '.xlsx')]"],
	'4': [
		"//a[contains(normalize-space(.), 'List of SLAs holding category A')]",
		"//a[contains(normalize-space(.), 'List of SLAs holding category B')]",
	],
	'5': [
		"//a[contains(@href, 'istoriilor') and (contains(@href, '.xlsx') or contains(@href, '.docx') or (contains(@href, '.doc') and not(contains(@href, '.docx'))))]",
		"//a[contains(@href, 'evidenta contractelor')]",
	],
}

topology = {
	regulatorName + ' 1': 'List of authorized banks of the Republic of Moldova',
	regulatorName + ' 2': 'List of authorized insurances of the Republic of Moldova',
	regulatorName + ' 3': 'List of Non-bank credit organizations of the Republic of Moldova',
	regulatorName + ' 4': 'List of Savings and Lending Associations of the Republic of Moldova',
	regulatorName + ' 5': 'List of Credit history bureaus of the Republic of Moldova',
}


sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
		  'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
		  'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
		  'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
		  'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
		  'Phone - Mother company': [], 'Check': []}

driver = webdriver.Chrome(options=chromeOptions)
driver.maximize_window()

In [6]:
#------------------------------------------------ Begin_Function ----------------------------------------
def pad(sqldict):
    n = len(sqldict['ListProcessDate'])
    for k in sqldict:
        sqldict[k] += [''] * (n - len(sqldict[k]))
    return sqldict

In [ ]:
COOKIE_XPATH = '//*[@id="block-bnm-privacy-bnm-privacy-block"]/div/div/div/div[1]/div[2]/span[1]'

def download_file(driver, xp, tempfolder, reg):
	"""Click an xpath link and wait for the file to land in tempfolder. Returns filepath or None."""
	before = set(os.listdir(tempfolder))
	el = WebDriverWait(driver, 30).until(EC.element_to_be_clickable((By.XPATH, xp)))
	href = urljoin(driver.current_url, (el.get_attribute('href') or '').split('#')[0].split('?')[0])
	driver.execute_script("arguments[0].scrollIntoView({block:'center'});", el)
	sleep(0.5)
	try:
		el.click()
	except Exception:
		driver.execute_script("arguments[0].click();", el)
	t0 = time.time()
	new = []
	while time.time() - t0 < 90:
		new = [f for f in set(os.listdir(tempfolder)) - before if not f.endswith(('.tmp', '.crdownload'))]
		if new:
			break
		sleep(0.5)
	if not new:
		print(f'[DL] {reg}: click failed — using GET {href[:80]}')
		driver.get(href)
		t0 = time.time()
		while time.time() - t0 < 90:
			new = [f for f in set(os.listdir(tempfolder)) - before if not f.endswith(('.tmp', '.crdownload'))]
			if new:
				break
			sleep(0.5)
	if not new:
		print(f'[WARN] {reg}: no file downloaded')
		return None
	print(f'[DL] {reg}: saved {new[0]}')
	return os.path.join(tempfolder, new[0])

def read_doc_table(fp):
	"""Convert .doc→.docx (or read .docx directly), return first table as DataFrame or None."""
	ext = os.path.splitext(fp)[1].lower()
	if ext == '.doc':
		docx_path = fp + 'x'
		word = win32.Dispatch("Word.Application")
		word.Visible = False
		word_doc = word.Documents.Open(fp)
		word_doc.SaveAs(docx_path, FileFormat=16)
		word_doc.Close()
		word.Quit()
		fp = docx_path
	doc = Document(fp)
	if not doc.tables:
		return None
	tbl = pd.DataFrame([[cell.text.strip() for cell in row.cells] for row in doc.tables[0].rows])
	tbl.columns = tbl.iloc[0]
	return tbl[1:].reset_index(drop=True)

def read_xlsx_lock(fp, min_rows=1):
	names = []
	with pd.ExcelFile(fp) as xlsx:
		for sn in xlsx.sheet_names:
			raw = pd.read_excel(xlsx, sheet_name=sn, header=None).dropna(how='all')
			if raw.shape[0] < min_rows:
				print(f'[SKIP] Sheet "{sn}" too small ({raw.shape[0]} rows)')
				continue
			mask = pd.to_numeric(raw.iloc[:, 0], errors='coerce').notna() & raw.iloc[:, 1].notna()
			body = raw[mask].copy().reset_index(drop=True)
			body['_name'] = body.iloc[:, 1].astype(str).str.strip()
			body = body[~body['_name'].str.contains(r'radia', case=False, na=False)]
			body = body[body['_name'].str.len() > 0]
			print(f'[INFO] Sheet "{sn}": {body.shape[0]} valid rows')
			names.extend(body['_name'].tolist())
	return names

# ======================== MAIN LOOP ========================
for reg in regdict:
	print(f'Working with {reg}.')
	list_code = reg.split(' ')[-1]

	driver.get(regdict[reg])
	sleep(3)
	try:
		cb = WebDriverWait(driver, 8).until(EC.element_to_be_clickable((By.XPATH, COOKIE_XPATH)))
		driver.execute_script("arguments[0].click();", cb)
		sleep(0.5)
	except TimeoutException:
		pass

	# ---- List 1: HTML scraping (banks) ----
	if list_code == '1':
		sleep(5)
		soup = BeautifulSoup(driver.page_source.replace('<br>', '***'), 'html.parser')
		banknames = soup.find_all('h3', {'class': 'bank-name'})
		sqldict['Name'].extend([ele.text.strip() for ele in banknames if len(ele.text.lower().replace('banks', '').strip()) > 1])
		maindiv = soup.find('div', {'class': 'container-post'})
		for div in maindiv.find_all('div', {'class': 'bank-content'})[1:]:
			sqldict['ListProcessDate'].append(processdate)
			sqldict['RegCtry'].append('MD')
			sqldict['RegCode'].append('NBMO')
			sqldict['ListCode'].append(list_code)
			sqldict['RegulationType'].append('Regulated')
			sqldict['ListName'].append(topology[reg])
			for line in div.find_all('div', {'class': 'line'}):
				linedivs = line.find_all('div')
				lbl = linedivs[0].text
				val = linedivs[1].text.strip()
				if 'Address' in lbl:   sqldict['Address_1'].append(val)
				if 'Phone' in lbl:     sqldict['Phone'].append(val)
				if 'Fax' in lbl:       sqldict['Fax'].append(val)
				if 'SWIFT' in lbl:     sqldict['BIC SWIFT Code'].append(val)
				if 'E-mail' in lbl:    sqldict['Email'].append(val)
				if 'WWW' in lbl:       sqldict['Website'].append(val)
			sqldict = pad(sqldict)

	# ---- List 2: xlsx files (insurers, brokers, bancassurance) ----
	elif list_code == '2':
		driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
		sleep(2)
		for index, xp in enumerate(xlsx_xpath['2']):
			fp = download_file(driver, xp, tempfolder, reg)
			if not fp: continue
			data_ = pd.read_excel(fp)
			if index == 0:
				mask1 = pd.to_numeric(data_.iloc[:, 0], errors='coerce').notna() | data_.iloc[:, 1].notna()
				mask2 = data_.iloc[:, 1].str.contains('radiată', case=False, na=False)
				mask3 = data_.iloc[:, 0].str.contains('nr', case=False, na=False)
				mask4 = data_.iloc[:, 1].notna()

				data_ = data_[mask1]
				data_ = data_[~mask2]
				data_ = data_[~mask3]
				data_ = data_[mask4]
				for _, row in data_.iterrows():
					name_val = str(row.iloc[1]).strip()
					if not name_val or len(name_val) == 0:
						continue
					name_val_clean = name_val.replace('„','').replace('”','').replace(',','')
					# print(name_val_clean)
					address_ = str(row.iloc[2]).strip() if pd.notna(row.iloc[2]) else ''
					address_clean = address_.split(';')[0].replace('\n',' ').replace('  ',' ').strip()
					# print(address_clean)
					zip_ = address_.split(',')[0]
					# print(zip_)
					sqldict['Name'].append(name_val)
					sqldict['Address_1'].append(address_clean.replace('\n',' '))
					sqldict['Zip'].append(zip_)
					sqldict['ListProcessDate'].append(processdate)
					sqldict['RegCtry'].append('MD')
					sqldict['RegCode'].append('NBMO')
					sqldict['ListCode'].append(list_code)
					sqldict['ListName'].append(topology[reg])
					sqldict['RegulationType'].append('Regulated')
				sqldict = pad(sqldict)
			elif index ==1:
				mask1 = pd.to_numeric(data_.iloc[:, 0], errors='coerce').notna() | data_.iloc[:, 1].notna()
				mask2 = data_.iloc[:, 1].str.contains('radiată', case=False, na=False)
				mask3 = data_.iloc[:, 0].str.contains('nr', case=False, na=False)
				mask4 = data_.iloc[:, 1].notna()

				data_ = data_[mask1]
				data_ = data_[~mask2]
				data_ = data_[~mask3]
				data_ = data_[mask4]
				for _, row in data_.iterrows():
					name_val = str(row.iloc[1]).strip()
					if not name_val or len(name_val) == 0:
						continue
					name_val_clean = name_val.replace('„','').replace('”','').replace(',','')
					# print(name_val_clean)
					idno_ = str(row.iloc[2]).strip() if pd.notna(row.iloc[2]) else ''
					address_ = str(row.iloc[3]).strip() if pd.notna(row.iloc[2]) else ''
					address_clean = address_.split(';')[0].replace('\n',' ').replace('  ',' ').strip()
					# print(address_clean)
					zip_ = address_.split(',')[0]
					# print(zip_)
					sqldict['Name'].append(name_val)
					sqldict['InternalID_1'].append(idno_)
					sqldict['InternalID_1_type'].append('IDNO')
					sqldict['Address_1'].append(address_clean.replace('\n',' '))
					sqldict['Zip'].append(zip_)
					sqldict['ListProcessDate'].append(processdate)
					sqldict['RegCtry'].append('MD')
					sqldict['RegCode'].append('NBMO')
					sqldict['ListCode'].append(list_code)
					sqldict['ListName'].append(topology[reg])
					sqldict['RegulationType'].append('Regulated')
				sqldict = pad(sqldict)	
			elif index == 2:
				mask1 = pd.to_numeric(data_.iloc[:, 0], errors='coerce').notna() | data_.iloc[:, 1].notna()
				mask2 = data_.iloc[:, 1].str.contains('radiată', case=False, na=False)
				mask3 = data_.iloc[:, 0].str.contains('nr', case=False, na=False)
				mask4 = data_.iloc[:, 1].notna()

				data_ = data_[mask1]
				data_ = data_[~mask2]
				data_ = data_[~mask3]
				data_ = data_[mask4]
				for _, row in data_.iterrows():
					name_val = str(row.iloc[2]).strip()
					if not name_val or len(name_val) == 0:
						continue
					name_val_clean = name_val.replace('„','').replace('”','').replace(',','')
					# print(name_val_clean)
					unique_reg_code = str(row.iloc[1]).strip() if pd.notna(row.iloc[0]) else ''
					idno_ = str(row.iloc[3]).strip() if pd.notna(row.iloc[2]) else ''
					address_ = str(row.iloc[4]).strip() if pd.notna(row.iloc[2]) else ''
					address_clean = address_.split(';')[0].replace('\n',' ').replace('  ',' ').strip()
					phone_ = str(row.iloc[5]).strip() if pd.notna(row.iloc[5]) else ''
					email_ = str(row.iloc[6]).strip() if pd.notna(row.iloc[6]) else ''

					# print(address_clean)
					zip_ = address_.split(',')[0]
					# print(zip_)
					sqldict['Name'].append(name_val)
					sqldict['InternalID_1'].append(idno_)
					sqldict['InternalID_1_type'].append('IDNO')
					sqldict['InternalID_2'].append(unique_reg_code)
					sqldict['InternalID_2_type'].append('The unique registration code in the Register')
					sqldict['Address_1'].append(address_clean.replace('\n',' '))
					sqldict['Zip'].append(zip_.replace('sediu:','').strip())
					sqldict['Phone'].append(phone_)
					sqldict['Email'].append(email_)
					sqldict['ListProcessDate'].append(processdate)
					sqldict['RegCtry'].append('MD')
					sqldict['RegCode'].append('NBMO')
					sqldict['ListCode'].append(list_code)
					sqldict['ListName'].append(topology[reg])
					sqldict['RegulationType'].append('Regulated')
				sqldict = pad(sqldict)	
	# ---- List 3: xlsx (non-bank credit orgs) ----
	elif list_code == '3':
		driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
		sleep(2)
		for xp in xlsx_xpath['3']:
			fp = download_file(driver, xp, tempfolder, reg)
			if not fp: continue
			data_ = pd.read_excel(fp)
			data_.columns = data_.iloc[0]
			data_ = data_[1:]

			mask1 = pd.to_numeric(data_.iloc[:, 0], errors='coerce').notna() | data_.iloc[:, 1].notna()
			mask2 = data_.iloc[:, 11].str.contains('Active', case=False, na=False)
			mask3 = data_.iloc[:, 0].str.contains('nr', case=False, na=False)
			mask4 = data_.iloc[:, 1].notna()


			data_ = data_[mask1]
			data_ = data_[mask2]
			data_ = data_[~mask3]
			data_ = data_[mask4]


			for _, row in data_.iterrows():
				name_val = str(row.iloc[1]).strip()
				if not name_val or len(name_val) == 0:
					continue
				name_val_clean = name_val.replace('„','').replace('”','').replace(',','')
				# print(name_val_clean)
				idno_ = str(row.iloc[3]).strip() if pd.notna(row.iloc[3]) else ''
				date_reg = str(row.iloc[4]).strip() if pd.notna(row.iloc[4]) else ''
				address_ = str(row.iloc[5]).strip() if pd.notna(row.iloc[5]) else ''
				address_clean = address_.split(';')[0].replace('\n',' ').replace('  ',' ').strip()
				# print(address_clean)
				zip_ = address_.split(',')[0]
				sqldict['Name'].append(name_val_clean.replace('\n',' '))
				sqldict['Address_1'].append(address_clean.replace('\n',' '))
				sqldict['InternalID_1'].append(idno_)
				sqldict['InternalID_1_type'].append('IDNO')
				
				sqldict['Zip'].append(zip_)
				sqldict['ListProcessDate'].append(processdate)
				sqldict['RegCtry'].append('MD')
				sqldict['RegCode'].append('NBMO')
				sqldict['ListCode'].append(list_code)
				sqldict['ListName'].append(topology[reg])
				sqldict['RegulationType'].append('Regulated')

	# ---- List 4: doc/docx (SLA category A + B) ----
	elif list_code == '4':
		driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
		sleep(2)
		for xp in xlsx_xpath['4']:
			fp = download_file(driver, xp, tempfolder, reg)
			if not fp: continue
			tbl = read_doc_table(fp)
			if tbl is None:
				print(f'[WARN] {reg}: no tables in doc')
				continue
			print(f'[INFO] Doc table: {tbl.shape}')
			data_  = tbl[1:]
			data_ = data_.drop_duplicates(subset=data_.columns[2], keep='first')
			for _, row in data_.iterrows():
				name_val = str(row.iloc[1]).strip() if pd.notna(row.iloc[1]) else ''
				if not name_val or re.match(r'^\d+\.?$', name_val):
					continue
				name_val_clean = name_val.split('\n')[0]
				idno_ = str(row.iloc[2]).strip() if pd.notna(row.iloc[2]) else ''
				address_ = str(row.iloc[3]).strip() if pd.notna(row.iloc[3]) else ''
				address_clean = address_.split(';')[0].replace('\n',' ').replace('  ',' ').strip()
				zip_ = address_clean.split(',')[0]
				phone_ = str(row.iloc[4]).strip() if pd.notna(row.iloc[-1]) else ''
				sqldict['Name'].append(name_val_clean)
				sqldict['InternalID_1'].append(idno_)
				sqldict['InternalID_1_type'].append('IDNO')
				sqldict['Address_1'].append(address_clean)
				sqldict['Zip'].append(zip_)
				# sqldict['Phone'].append(phone_)
				sqldict['ListProcessDate'].append(processdate)
				sqldict['RegCtry'].append('MD')
				sqldict['RegCode'].append('NBMO')
				sqldict['ListCode'].append(list_code)
				sqldict['ListName'].append(topology[reg])
				sqldict['RegulationType'].append('Regulated')
			sqldict = pad(sqldict)

	# ---- List 5: .doc (credit history bureaus) + .xlsx (contract sources) ----
	elif list_code == '5':
		driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
		sleep(2)
		for xp in xlsx_xpath['5']:
			fp = download_file(driver, xp, tempfolder, reg)
			if not fp: continue
			ext = os.path.splitext(fp)[1].lower()

			if ext in ('.doc', '.docx'):
				tbl = read_doc_table(fp)
				if tbl is None:
					print(f'[WARN] {reg}: no tables in doc')
					continue
				print(f'[INFO] Doc table: {tbl.shape}')
				for _, row in tbl.iterrows():
					name_val = str(row.iloc[1]).strip() if pd.notna(row.iloc[1]) else ''
					if not name_val or re.match(r'^\d+\.?$', name_val):
						continue
					name_val_clean = name_val.split('\n')[0]
					idno_ = str(row.iloc[2]).strip() if pd.notna(row.iloc[2]) else ''
					address_ = str(row.iloc[3]).strip() if pd.notna(row.iloc[3]) else ''
					address_clean = address_.split(';')[4].replace('\n',' ').replace('  ',' ').strip()
					zip_ = address_clean.split(',')[0]
					
					sqldict['Name'].append(name_val_clean)
					sqldict['InternalID_1'].append(idno_)
					sqldict['InternalID_1_type'].append('IDNO')
					sqldict['Address_1'].append(address_clean)
					sqldict['Zip'].append(zip_)
					sqldict['Phone'].append(phone_.split('\n')[0])
					sqldict['ListProcessDate'].append(processdate)
					sqldict['RegCtry'].append('MD')
					sqldict['RegCode'].append('NBMO')
					sqldict['ListCode'].append(list_code)
					sqldict['ListName'].append(topology[reg])
					sqldict['RegulationType'].append('Regulated')

				sqldict = pad(sqldict)

			elif ext in ('.xlsx', '.xls'):
				data_ = pd.read_excel(fp)
				mask1 = pd.to_numeric(data_.iloc[:, 0], errors='coerce').notna() | data_.iloc[:, 1].notna()
				mask2 = data_.iloc[:, 1].str.contains('radiată|retrasa|suspendată', case=False, na=False)
				# mask3 = data_.iloc[:, 0].str.contains('nr', case=False, na=False)
				mask4 = data_.iloc[:, 0].notna()
				mask5 = data_.iloc[:, 1].notna()

				data_ = data_[mask1]
				data_ = data_[~mask2]
				# data_ = data_[~mask3]
				data_ = data_[mask4]
				data_ = data_[mask5]
				for _, row in data_.iterrows():
					name_val = str(row.iloc[1]).strip()
					if not name_val or len(name_val) == 0:
						continue
					name_val_clean = name_val.replace('„','').replace('”','').replace(',','')
					sqldict['Name'].append(name_val_clean)
					sqldict['ListProcessDate'].append(processdate)
					sqldict['RegCtry'].append('MD')
					sqldict['RegCode'].append('NBMO')
					sqldict['ListCode'].append(list_code)
					sqldict['ListName'].append(topology[reg])
					sqldict['RegulationType'].append('Regulated')
				sqldict = pad(sqldict)

	for rem in os.listdir(tempfolder):
		os.remove(os.path.join(tempfolder, rem))


Working with MD NBMO 1.
Working with MD NBMO 2.
[DL] MD NBMO 2: saved RPPPA_Asiguratori_BNM 24_03_2026.xlsx
[DL] MD NBMO 2: saved Registrul brokerilor de asigurare (reasigurare)_20.xlsx
[DL] MD NBMO 2: saved Registrul agenților de asigurare și agenților bancassurance_13.xlsx
Working with MD NBMO 3.
[DL] MD NBMO 3: saved ROCNA_BNM_web_43.xlsx
Working with MD NBMO 4.
[DL] MD NBMO 4: saved AEÎ ce dețin licențe de categoria A_10.docx
[INFO] Doc table: (132, 7)
[DL] MD NBMO 4: saved AEI ce detin licențe de categoria B si licența ANC_3.docx
[INFO] Doc table: (111, 7)
Working with MD NBMO 5.
[DL] MD NBMO 5: saved Lista Birourilor istoriilor de credit.doc
[INFO] Doc table: (4, 6)
[DL] MD NBMO 5: saved evidenta contractelor surselor cu BIC_8.xlsx


In [8]:
#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------
os.chdir(scriptfolder)
df=pd.DataFrame(sqldict)
df.to_excel(filename, 'SQL Ready', index=False)
driver.quit()
sleep(3)

C:\Users\wuj1\AppData\Local\Temp\1\ipykernel_53648\2462146938.py:4: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(filename, 'SQL Ready', index=False)


In [10]:
import os
import re
import pandas as pd
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas

def safe_filename(name):
    name = str(name).strip()
    return re.sub(r'[\\/:*?"<>|]+', '_', name) or "UNKNOWN"
def export_list_to_pdf(data_list, pdf_filename):
    c = canvas.Canvas(pdf_filename, pagesize=letter)
    c.setFont("Helvetica", 12)

    x, y = 72, 720
    max_lines_per_page = 33
    line_count = 0

    for item in data_list:
        c.drawString(x, y, str(item))
        y -= 20
        line_count += 1

        if line_count >= max_lines_per_page:
            c.showPage()
            c.setFont("Helvetica", 12)
            x, y = 72, 720
            line_count = 0

    c.save()

os.makedirs(tempfolder, exist_ok=True)

for list_code, group in df.groupby('ListCode', dropna=False):
    code = safe_filename(list_code)
    items = group['Name'].dropna().astype(str).tolist()
    if not items:
        continue
    pdf_path = os.path.join(tempfolder, f"{filename}- {code}.pdf")
    export_list_to_pdf(items, pdf_path)